# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Usaf007/flyrankai-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Distribution Analysis of Key Signals**

Before testing specific hypotheses, we need to observe the raw distributions of our numeric features. In search data, we expect metrics like `gsc_impressions` and `gsc_clicks` to exhibit heavy right-tails (power-law distribution), where a small fraction of URLs capture the vast majority of traffic. We will pull a sample from our mid-panel month (March 2026) to verify this.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata

# Authenticate and setup duckdb connection
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
dim_table = f"{rel}/dim_content.parquet"

# Fetch a representative sample to check distributions and heavy tails
dist_query = f"""
    SELECT
        f.gsc_impressions,
        f.gsc_clicks,
        f.ga4_sessions,
        d.word_count,
        date_diff('day', d.content_created_date, f.report_date) AS age_days
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 100000
"""
df_dist = con.execute(dist_query).df()

# Display standard distribution metrics (percentiles emphasize the heavy tails in traffic)
display(df_dist.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,ga4_sessions,word_count,age_days
count,100000.00,100000.00,100000.00,95986.0,100000.00
mean,220.87,1.04,2.53,3235.29,157.73
std,478.99,3.04,5.68,1249.19,115.54
min,0.00,0.00,0.00,8.0,0.00
25%,13.00,0.00,1.00,2615.0,51.00
50%,81.00,0.00,1.00,3058.0,139.00
75%,234.00,1.00,2.00,3653.0,215.00
90%,542.10,3.00,5.00,4966.5,346.00
99%,2127.00,9.00,22.00,7106.3,447.00
max,16902.00,260.00,361.00,9981.0,473.00


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1: Word Count Depth**
*   **Verdict: OPPOSITE.**
*   **Why:** I assumed longer content was safer and more authoritative. The data shows the exact opposite: Short content (<1k words) only decays at a 9.5% rate, while Long content (3k+ words) decays at a massive 48.09%. In this dataset, longer content is significantly more vulnerable to losing its page-one rankings.

**Signal 2: Initial Traffic Volume**
*   **Verdict: FALSE.**
*   **Why:** I assumed high-volume pages might be insulated from decay. However, the decay rates across Low, Mid, and High volume tiers are virtually identical (ranging from 26.45% to 27.88%). Raw daily impression volume has essentially zero impact on a page's probability of dropping off page one.

**Signal 3: Age Stability**
*   **Verdict: OPPOSITE.**
*   **Why:** The heuristic rule often assumes older content decays more. The data shows that content hits peak volatility during its "Maturing" phase (6-12 months) with a 37.03% decay rate. But once it survives past a year (Stale), its decay rate plummets to just 11.25%. Older content is actually much safer.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Testing the three signals against the decay target (avg_position > 10)
signal_tests_query = f"""
    SELECT
        -- Signal 1: Word Count Bucket
        CASE
            WHEN d.word_count < 1000 THEN '1. Short (<1k)'
            WHEN d.word_count < 3000 THEN '2. Medium (1k-3k)'
            ELSE '3. Long (3k+)'
        END AS word_count_tier,

        -- Signal 2: Volume Bucket
        CASE
            WHEN f.gsc_impressions < 100 THEN '1. Low Vol'
            WHEN f.gsc_impressions < 1000 THEN '2. Mid Vol'
            ELSE '3. High Vol'
        END AS volume_tier,

        -- Signal 3: Age Bucket
        CASE
            WHEN date_diff('day', d.content_created_date, f.report_date) < 180 THEN '1. Fresh (<6mo)'
            WHEN date_diff('day', d.content_created_date, f.report_date) < 365 THEN '2. Maturing (6-12mo)'
            ELSE '3. Stale (12mo+)'
        END AS age_tier,

        COUNT(*) as row_count,
        ROUND(AVG(CASE WHEN f.gsc_avg_position > 10 THEN 1.0 ELSE 0.0 END) * 100, 2) AS decay_rate_pct
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1, 2, 3
"""
df_signals = con.execute(signal_tests_query).df()

# Aggregate and display the tests individually for clear review
display(df_signals.groupby('word_count_tier')[['row_count', 'decay_rate_pct']].mean().round(2))
display(df_signals.groupby('volume_tier')[['row_count', 'decay_rate_pct']].mean().round(2))
display(df_signals.groupby('age_tier')[['row_count', 'decay_rate_pct']].mean().round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,decay_rate_pct
word_count_tier,,
1. Short (<1k),590.50,9.51
2. Medium (1k-3k),20783.33,22.51
3. Long (3k+),24688.00,48.09


,row_count,decay_rate_pct
volume_tier,,
1. Low Vol,25910.56,26.45
2. Mid Vol,18438.89,27.82
3. High Vol,1852.62,27.88


,row_count,decay_rate_pct
age_tier,,
1. Fresh (<6mo),30909.50,34.61
2. Maturing (6-12mo),14744.00,37.03
3. Stale (12mo+),3777.11,11.25


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-Linked Test: CTR Underperformance (The "CTR-Fix" Flag)**
*   **Verdict: FALSE (Flawed Premise).**
*   **Why:** The test returned a 0.0% decay rate across all CTR health buckets. This reveals a structural leakage error in trying to prove this flag using same-day data: because we filtered the query to only look at pages currently ranking in positions 1-5, it is mathematically impossible for those exact same rows to simultaneously register an average position drop > 10. This proves that same-day warehouse snapshots cannot validate predictive flags like CTR without introducing a time lag (e.g., matching last week's CTR to this week's position).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Isolating pages already ranking well (Positions 1-5) to test if poor CTR predicts a drop
ctr_flag_query = f"""
    SELECT
        CASE
            WHEN (CAST(f.gsc_clicks AS FLOAT) / NULLIF(f.gsc_impressions, 0)) < 0.01 THEN '1. Danger: CTR < 1%'
            WHEN (CAST(f.gsc_clicks AS FLOAT) / NULLIF(f.gsc_impressions, 0)) < 0.03 THEN '2. Warning: CTR 1-3%'
            ELSE '3. Healthy: CTR 3%+'
        END AS ctr_health,
        COUNT(*) as observed_days,
        ROUND(AVG(CASE WHEN f.gsc_avg_position > 10 THEN 1.0 ELSE 0.0 END) * 100, 2) AS decay_rate_pct
    FROM read_parquet('{fact_table}') f
    WHERE f.ga4_data_available IS TRUE
      AND f.gsc_avg_position <= 5
      AND f.gsc_impressions > 50
    GROUP BY 1
    ORDER BY 1
"""
df_ctr_test = con.execute(ctr_flag_query).df()
display(df_ctr_test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_health,observed_days,decay_rate_pct
0,1. Danger: CTR < 1%,45271,0.0
1,2. Warning: CTR 1-3%,18110,0.0
2,3. Healthy: CTR 3%+,2220,0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Conclusion for Content Operations**

For the content team, these measured signals demand a strategy pivot. Word count is not a defensive moat (long-form content decays the fastest), and age is a stabilizer, not a risk factor. Furthermore, attempting to use behavioral metrics like CTR as an early-warning signal requires a time-aware reporting architecture; trying to diagnose future decay using same-day performance dashboards will blindly mask the drop.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.